# Multi-Index RAG
### Querying multiple heterogeneous collections and merging results

Two knowledge bases that differ in structure and update cadence — exactly the case Multi-Index RAG is for:
- **Index A** = `OWASP Top 10 for LLMs (2025)` — 10 named risk categories, on **Pinecone** (managed, cloud).
- **Index B** = `NIST AI RMF (AI.100-1)` — a broader risk-management framework, on **FAISS** (local, in-process).

## Step 1: Build Index A — OWASP on Pinecone

In [1]:
!pip install langchain langchain-community langchain-openai langchain-pinecone pinecone langchain-text-splitters faiss-cpu pypdf python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, time
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

owasp_chunks = splitter.split_documents(PyPDFLoader("OWASP-Top-10-for-LLMs-v2025.pdf").load())

INDEX_NAME = "adv-rag-owasp"
pc = Pinecone()

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,  # text-embedding-3-small output size
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

index_a = PineconeVectorStore.from_documents(owasp_chunks, embedding=embeddings, index_name=INDEX_NAME)
print(f"Index A (Pinecone): {len(owasp_chunks)} OWASP chunks upserted")

C:\Users\shiva\AppData\Local\Temp\ipykernel_22572\1660417976.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Index A (Pinecone): 131 OWASP chunks upserted


## Step 2: Build Index B — NIST on FAISS
Same embedding model as Index A, so results are comparable at the merge layer.

In [3]:
nist_chunks = splitter.split_documents(PyPDFLoader("NIST.AI.100-1.pdf").load())
index_b = FAISS.from_documents(nist_chunks, embeddings)
print(f"Index B (FAISS): {len(nist_chunks)} NIST chunks -> {index_b.index.ntotal} vectors")

Index B (FAISS): 150 NIST chunks -> 150 vectors


## Step 3: Parallel retrieval across both indexes
Each index is searched independently with the same query — this is the "parallel retrieval across all indexes" step from the architecture diagram.

In [4]:
def search_all(query, k=4):
    results_a = index_a.similarity_search(query, k=k)
    for d in results_a:
        d.metadata["retriever"] = "OWASP (Pinecone)"
    results_b = index_b.similarity_search(query, k=k)
    for d in results_b:
        d.metadata["retriever"] = "NIST (FAISS)"
    return results_a, results_b

## Step 4: Merge layer — combine, dedupe, rank
Pinecone reports cosine similarity and FAISS reports L2 distance — the raw scores from the two indexes are **not on the same scale**, so ranking by raw score would silently favor one index. The standard fix is **Reciprocal Rank Fusion (RRF)**: fuse by each document's *rank position* within its own list, not by its raw score.

In [5]:
def rrf_merge(*ranked_lists, k=60):
    scores = {}
    docs = {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked, start=1):
            key = (doc.metadata["retriever"], doc.metadata.get("page"), doc.page_content[:60])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    fused = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [docs[key] for key, _ in fused]

## Step 5: Full pipeline — retrieve from both, merge, generate

In [6]:
RAG_PROMPT = """Answer the question using only the following context. Note which source (OWASP or NIST) each part of your answer comes from.

Context:
{context}

Question: {query}
Answer:"""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def answer(query, k=4, top_n=6):
    results_a, results_b = search_all(query, k=k)
    merged = rrf_merge(results_a, results_b)[:top_n]
    context = "\n\n".join(f"[{d.metadata['retriever']}] {d.page_content}" for d in merged)
    response = llm.invoke(RAG_PROMPT.format(context=context, query=query)).content.strip()
    return merged, response

## Step 6: Try a query that genuinely needs both indexes
A single-source retriever can only ever surface facts from its own collection. Compare Index A alone, Index B alone, and the merged result.

In [7]:
query = "How do OWASP's LLM Top 10 risk categories map onto the core functions of NIST's AI Risk Management Framework?"

only_a, only_b = search_all(query, k=4)
merged, response = answer(query)

print("Index A alone (OWASP/Pinecone) sources:", [d.metadata["page"] for d in only_a])
print("Index B alone (NIST/FAISS) sources:    ", [d.metadata["page"] for d in only_b])
print("Merged, top", len(merged), "sources:        ", [(d.metadata["retriever"], d.metadata["page"]) for d in merged])
print("\nAnswer:\n", response)

Index A alone (OWASP/Pinecone) sources: [4.0, 16.0, 13.0, 17.0]
Index B alone (NIST/FAISS) sources:     [7, 0, 43, 24]
Merged, top 6 sources:         [('OWASP (Pinecone)', 4.0), ('NIST (FAISS)', 7), ('OWASP (Pinecone)', 16.0), ('NIST (FAISS)', 0), ('OWASP (Pinecone)', 13.0), ('NIST (FAISS)', 43)]

Answer:
 The OWASP Top 10 for LLM Applications identifies specific security issues related to AI applications, while NIST's AI Risk Management Framework (AI RMF) provides a structured approach to managing risks associated with AI systems. 

1. **GOVERN**: This function applies to all stages of AI risk management processes and procedures. OWASP's emphasis on raising awareness and building a foundation for secure LLM usage aligns with the governance aspect of ensuring that security practices are integrated throughout the AI lifecycle. (Source: OWASP)

2. **MAP**: This function involves identifying and assessing risks specific to AI systems. OWASP's recommendation to apply comprehensive AI Red T

## Try it yourself
1. Add a third index (e.g. a company policy PDF) and extend `search_all` / `rrf_merge` to three lists.
2. Ask a purely single-source question and confirm the merge layer still ranks that index's chunks on top.
3. Swap RRF for a naive "sort by raw score" merge and see how it skews toward whichever index's metric happens to produce larger numbers.

**Cleanup:** this created a live Pinecone index (`adv-rag-owasp`). Free tier covers it, but you can remove it with `pc.delete_index(INDEX_NAME)` when done.